# Indexing

In `xftsim` we frequently want to refer to specific individuals,
specific variants, or specific phenotypic components. There are two
parallel systems for this in v0.9:

- **The `xftsim.index` module** — `SampleIndex`,
  `DiploidVariantIndex` / `HaploidVariantIndex`, and `ComponentIndex`.
  These are pandas-DataFrame-backed objects with friendly constructors
  and inter-conversion. They're still useful when you want a tabular
  view or are interfacing with external data (PLINK, sgkit).
- **The `xftsim.struct` metadata dataclasses** — `SampleMeta` and
  `VariantMeta`. These are lighter immutable numpy-backed dataclasses
  that ride along with `DenseHaplotypeArray` and `PhenotypeArray`.
  These are what the simulator actually carries between generations.

In day-to-day v0.9 use you'll rarely construct an index object by
hand — the formula DSL refers to phenotypes by name (`'height.G'`,
`'wealth'`) instead of by `ComponentIndex`, and founder data
constructors create the appropriate metadata automatically. This
tutorial is mostly here for completeness and when you need to
interoperate with the legacy interfaces.

## Component indexing (`xftsim.index.ComponentIndex`)

The most important indexer is `ComponentIndex`. An individual-level
component is described by:

- `phenotype_name` — e.g. `'height'`, `'BMD'`
- `component_name` — e.g. `'additiveGenetic'`, `'additiveNoise'`,
  `'phenotype'`
- `vorigin_relative` — which generation/relation the component refers
  to (proband, mother, father, …)
- `comp_type` — `'outcome'` for the final phenotype, `'intermediate'`
  for a subcomponent


In [ ]:
import xftsim as xft

cindex = xft.index.ComponentIndex.from_product(
    ('phenotype_1', 'phenotype_2'),
    ('additiveGenetic', 'additiveNoise', 'phenotype'),
)
cindex.frame


`vorigin_relative` encodes the relationship to a proband:

| `vorigin_relative` | relationship |
| --- | --- |
| -1 | self |
|  0 | mother |
|  1 | father |
| 00 | maternal grandmother |
| 10 | maternal grandfather |
| 01 | paternal grandmother |
| 11 | paternal grandfather |
| 000 | maternal grandmother's mother |
| … | … |

In v0.9 you rarely set `vorigin_relative` directly — the DSL
`mother(...)` / `father(...)` / `parent(...)` builtins (see the
architectures tutorial) handle the cross-generation lookup for you.

### Constructing component indices

From a Cartesian product:


In [ ]:
xft.index.ComponentIndex.from_product(
    phenotype_name=('height',),
    component_name=('phenotype',),
    vorigin_relative=(-1, 1),
)


Explicitly:

In [ ]:
cindex = xft.index.ComponentIndex(
    phenotype_name=('height', 'BMD'),
    component_name=('phenotype', 'genetic'),
    vorigin_relative=(-1, 1),
)
cindex


From an existing DataFrame:

In [ ]:
xft.index.ComponentIndex(frame=cindex.frame)

Or for a generic `k`-trait index without naming traits:


In [ ]:
xft.index.ComponentIndex(k_total=3)

## Variant indexing

`xftsim.index.DiploidVariantIndex` and `HaploidVariantIndex` track:

- `vid`, `chrom`
- `zero_allele`, `one_allele`
- `af` — ancestral allele frequencies
- `pos_bp`, `pos_cM`
- `annotation_array`, `h_copy`

You very rarely build one by hand — founder constructors and PLINK / VCF
importers fill these in for you.

### Constructing variant indices


In [ ]:
vind = xft.index.DiploidVariantIndex(m=500, n_chrom=22)
vind


In [ ]:
xft.index.DiploidVariantIndex(vid=vind.vid)

In [ ]:
xft.index.DiploidVariantIndex(frame=vind.frame)

### Switching between haploid and diploid


In [ ]:
hvind = vind.to_haploid()
hvind


In [ ]:
hvind.to_diploid()

## Sample indexing

`xftsim.index.SampleIndex` tracks `iid`, `fid`, `sex`, and `generation`:


In [ ]:
xft.index.SampleIndex(n=5, generation=1)

In [ ]:
sind = xft.index.SampleIndex(
    iid=['0_sister1', '0_sister2', '0_sister3', '0_brother1', '0_brother2'],
    fid=[0, 0, 0, 1, 1],
    sex=[0, 0, 0, 1, 1],
    generation=0,
)
sind


In [ ]:
xft.index.SampleIndex(frame=sind.frame, generation=1)

## The struct-level alternative: `SampleMeta` and `VariantMeta`

The new `Simulation` carries `xftsim.struct.SampleMeta` and
`xftsim.struct.VariantMeta` dataclasses on every haplotype array.
These are immutable, numpy-backed, and don't depend on pandas. You can
get them off any `DenseHaplotypeArray`:


In [ ]:
hap = xft.founders.founder_haplotypes_uniform_AFs(n=10, m=20)
print(hap.samples)
print(hap.variants)


For most v0.9 simulations this is all you need — the formula DSL
addresses phenotype components by string keys and the mating regimes
take a `component_names=[...]` list of strings, so `ComponentIndex`
construction never enters the picture.
